# 🛰️ 🌑 Análisis Sentinel-5P (NO₂) durante el Eclipse Solar en España, 12 de agosto de 2026

**Autora:** Lavinia Bacaru ([@codinglavinia](https://github.com/codinglavinia))
**Proyecto:** [Eclipse-2026](https://github.com/codinglavinia/Eclipse-2026)

## 🎯 Objetivo

Estudiar la evolución del **dióxido de nitrógeno (NO₂)** troposférico sobre España
antes, durante y después del eclipse solar total del 12 de agosto de 2026, utilizando
datos del satélite **Sentinel-5P (instrumento TROPOMI)**.

Se analizan tres fechas:
- 🌤️ **11 de agosto** — antes del eclipse
- 🌑 **12 de agosto** — día del eclipse
- 🌤️ **13 de agosto** — después del eclipse

## 🗂️ Datos utilizados

| Fuente | Formato | Contenido |
|---|---|---|
| Xjubier / EclipseCity | KMZ/KML | Trayectoria de la umbra y penumbra del eclipse |
| Copernicus Sentinel-5P | NetCDF (.nc) | Producto oficial L2 NO₂ (TROPOMI) |
| Copernicus Data Space | GeoTIFF (.tiff) | Producto NO₂ recortado (Raw) |

## ⚠️ Requisitos de ejecución

Este notebook está diseñado para ejecutarse en **Google Colab** y utiliza:
- `google.colab.files` para subir archivos manualmente
- `google.colab.drive` para montar Google Drive

Para reproducirlo, necesitarás tus propios archivos Sentinel-5P descargados desde el
[Copernicus Data Space](https://dataspace.copernicus.eu/) y el KMZ del eclipse desde
[Xjubier Eclipse Maps](http://xjubier.free.fr/).

> 💡 **Nota:** el notebook combina exploración inicial (carga manual de archivos) con
> el análisis final reproducible (vía Google Drive). Las secciones están marcadas
> claramente a continuación.


In [ ]:
!pip install geopandas fiona shapely pyogrio matplotlib contextily


## 1️⃣ Geometría del eclipse (KMZ / KML)

Carga y visualización de la trayectoria de la umbra y penumbra del eclipse solar.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

kmz_file = next(iter(uploaded))

print("Fichero KMZ:", kmz_file)
print("Tamaño fichero:", round(os.path.getsize(kmz_file) / 1024, 2), "KB")

with zipfile.ZipFile(kmz_file, "r") as kmz:
    print("\nContenido KMZ:")
    for file in kmz.namelist():
        print(" -", file)

In [ ]:
import zipfile
import os

# Nombre del archivo KML que hemos encontrado dentro del KMZ
nombre_kml = "TSE_2026_08_12__51d81c738c298.kml"

# Abrimos el archivo KMZ como un archivo ZIP
with zipfile.ZipFile(kmz_file, "r") as archivo_kmz:

    # Extraemos únicamente el archivo KML
    archivo_kmz.extract(nombre_kml, "/content")

# Definimos la ruta completa del archivo KML extraído
ruta_kml = f"/content/{nombre_kml}"

# Comprobamos que la extracción se ha realizado correctamente
print("KML extraído correctamente.")
print(f"📄 Archivo: {nombre_kml}")
print(f"📂 Ruta: {ruta_kml}")
print(f"🔎 ¿El archivo existe?: {os.path.exists(ruta_kml)}")

In [ ]:

import xml.etree.ElementTree as ET

# Cargamos el archivo KML
arbol = ET.parse(ruta_kml)

# Obtenemos el elemento raíz del documento
raiz = arbol.getroot()

# Mostramos información básica sobre el documento
print("Archivo KML leído correctamente.")
print(f"📌 Elemento raíz: {raiz.tag}")

# Contamos los elementos principales del KML
numero_placemarks = 0
numero_puntos = 0
numero_lineas = 0
numero_poligonos = 0

# Recorremos todos los elementos del documento
for elemento in raiz.iter():

    # Eliminamos el identificador de espacio de nombres
    etiqueta = elemento.tag.split("}")[-1]

    if etiqueta == "Placemark":
        numero_placemarks += 1

    elif etiqueta == "Point":
        numero_puntos += 1

    elif etiqueta == "LineString":
        numero_lineas += 1

    elif etiqueta == "Polygon":
        numero_poligonos += 1

# Mostramos el resumen de las geometrías encontradas
print("\n📊 RESUMEN DEL KML")
print("-" * 40)
print(f"📍 Placemark:   {numero_placemarks}")
print(f"📌 Puntos:      {numero_puntos}")
print(f"📏 Líneas:      {numero_lineas}")
print(f"🔷 Polígonos:   {numero_poligonos}")

In [ ]:
#identificar elementos del archivo kml

import xml.etree.ElementTree as ET

# Definimos el espacio de nombres utilizado por KML
espacio_kml = {"kml": "http://www.opengis.net/kml/2.2"}

# Buscamos todos los elementos Placemark
placemarks = raiz.findall(".//kml:Placemark", espacio_kml)

print(f"📌 Se han encontrado {len(placemarks)} elementos.\n")

# Recorremos cada Placemark y mostramos su información
for numero, placemark in enumerate(placemarks, start=1):

    # Buscamos el nombre del elemento
    elemento_nombre = placemark.find("kml:name", espacio_kml)

    # Obtenemos el texto del nombre
    if elemento_nombre is not None and elemento_nombre.text:
        nombre = elemento_nombre.text.strip()
    else:
        nombre = "Sin nombre"

    # Detectamos el tipo de geometría
    tipos_geometria = []

    if placemark.find(".//kml:Point", espacio_kml) is not None:
        tipos_geometria.append("Punto")

    if placemark.find(".//kml:LineString", espacio_kml) is not None:
        tipos_geometria.append("Línea")

    if placemark.find(".//kml:Polygon", espacio_kml) is not None:
        tipos_geometria.append("Polígono")

    # Unimos los tipos de geometría encontrados
    geometria = ", ".join(tipos_geometria) if tipos_geometria else "Sin geometría"

    # Mostramos la información
    print(f"{numero:02d}. {nombre}")
    print(f"    └── Geometría: {geometria}")

In [ ]:
# cargar geometrias del ECLIPSE
# Proyecto: Análisis del Eclipse Solar de España 2026
# Fuente: Xjubier / Eclipse

import geopandas as gpd

# Intentamos leer directamente el archivo KML con GeoPandas
gdf_eclipse = gpd.read_file(
    ruta_kml,
    driver="KML"
)

# Mostramos información general del GeoDataFrame
print("KML cargado correctamente con GeoPandas.")
print()

print("📊 Información del dataset:")
print(f"   Número de elementos: {len(gdf_eclipse)}")
print(f"   Sistema de coordenadas: {gdf_eclipse.crs}")
print()

# Mostramos las primeras filas
display(gdf_eclipse.head())

In [ ]:
# identificar todas las capas del dataset KML
import pyogrio

# Obtenemos la lista de capas internas del archivo KML
capas_kml = pyogrio.list_layers(ruta_kml)

print("🗺️ CAPAS ENCONTRADAS EN EL KML")
print("=" * 60)

# Mostramos cada capa y su tipo de geometría
for numero, capa in enumerate(capas_kml, start=1):

    nombre_capa = capa[0]
    tipo_geometria = capa[1]

    print(f"{numero}. {nombre_capa}")
    print(f"   └── Geometría: {tipo_geometria}")
    print()

In [ ]:
# CARGAR LAS CAPAS PRINCIPALES DEL ECLIPSE
# Proyecto: Análisis del Eclipse Solar de España 2026
# Fuente: Xjubier / EclipseCity


import geopandas as gpd

# ------------------------------------------------------------
# 1. PUNTO DE MÁXIMO DEL ECLIPSE
# ------------------------------------------------------------

gdf_maximo = gpd.read_file(
    ruta_kml,
    layer="Greatest Eclipse Point",
    driver="KML"
)

# ------------------------------------------------------------
# 2. ZONA DE LA UMBRA
# ------------------------------------------------------------

gdf_umbra = gpd.read_file(
    ruta_kml,
    layer="TSE 2026 August 12 Umbral Path",
    driver="KML"
)

# ------------------------------------------------------------
# 3. LÍMITES DE LA PENUMBRA
# ------------------------------------------------------------

gdf_penumbra = gpd.read_file(
    ruta_kml,
    layer="TSE 2026 August 12 Penumbral Limits",
    driver="KML"
)

# ------------------------------------------------------------
# MOSTRAR INFORMACIÓN DE LAS CAPAS
# ------------------------------------------------------------

print(" Capas cargadas correctamente.\n")

print("📍 Greatest Eclipse Point:")
print(f"   Elementos: {len(gdf_maximo)}")
print(f"   CRS: {gdf_maximo.crs}")
print(f"   Geometría: {gdf_maximo.geometry.iloc[0].geom_type}\n")

print("🌑 Umbral Path:")
print(f"   Elementos: {len(gdf_umbra)}")
print(f"   CRS: {gdf_umbra.crs}")
print(f"   Geometría: {gdf_umbra.geometry.iloc[0].geom_type}\n")

print("☀️ Penumbral Limits:")
print(f"   Elementos: {len(gdf_penumbra)}")
print(f"   CRS: {gdf_penumbra.crs}")
print(f"   Geometría: {gdf_penumbra.geometry.iloc[0].geom_type}")

In [ ]:
# 1ª VISUALIZACIÓN DEL ECLIPSE
# Proyecto: Análisis del Eclipse Solar de España 2026
# Fuente: Xjubier / EclipseCity


import matplotlib.pyplot as plt

# Creamos una figura para representar las geometrías
fig, ax = plt.subplots(figsize=(14, 9))

# ------------------------------------------------------------
# 1. representar la trayectoria de la umbra
# ------------------------------------------------------------

gdf_umbra.plot(
    ax=ax,
    linewidth=2,
    label="Trayectoria de la umbra"
)

# ------------------------------------------------------------
# 2. representar los limites de la PENUMBRA
# ------------------------------------------------------------

gdf_penumbra.plot(
    ax=ax,
    linewidth=1.5,
    linestyle="--",
    label="Límites de la penumbra"
)

# ------------------------------------------------------------
# 3. representar el punto maximo del ECLIPSE
# ------------------------------------------------------------

gdf_maximo.plot(
    ax=ax,
    markersize=80,
    marker="*",
    label="Máximo del eclipse"
)

# ------------------------------------------------------------
# Configuración del mapa
# ------------------------------------------------------------

ax.set_title(
    "Eclipse Solar Total — 12 de agosto de 2026",
    fontsize=16
)

ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")

ax.legend()

ax.grid(True, linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

## 2️⃣ Exploración inicial — Sentinel-5P NetCDF (carga manual)

Primera exploración de un producto Sentinel-5P NO₂ subido manualmente. Esta sección sirvió para entender la estructura del producto (grupo `PRODUCT`, variables, QA).

In [ ]:

# VERIFICAR ARCHIVOS SENTINEL-5P

import os

# Buscamos todos los archivos NetCDF (.nc) disponibles
archivos_nc = [
    archivo
    for archivo in os.listdir("/content")
    if archivo.lower().endswith(".nc")
]

# Mostramos los archivos encontrados
print("🛰️ ARCHIVOS SENTINEL-5P DISPONIBLES")
print("=" * 60)

if archivos_nc:
    for numero, archivo in enumerate(archivos_nc, start=1):
        print(f"{numero}. {archivo}")
else:
    print("❌ No se encontró ningún archivo .nc en /content/")

In [ ]:
# ============================================================
# CARGAR EL PRODUCTO SENTINEL-5P en el cuaderno GOOGLE COLAB
# Proyecto: Análisis del Eclipse Solar de España 2026
# ============================================================

from google.colab import files
import os

# Abrimos la ventana para seleccionar el archivo desde el ordenador
archivos_subidos = files.upload()

print("\n🛰️ ARCHIVOS CARGADOS")
print("=" * 60)

# Mostramos los archivos que se han cargado
for nombre_archivo in archivos_subidos:
    ruta_archivo = os.path.join("/content", nombre_archivo)

    print(f"📄 Archivo: {nombre_archivo}")
    print(f"📂 Ruta: {ruta_archivo}")
    print(f"✅ Existe: {os.path.exists(ruta_archivo)}")

In [ ]:

# Satélite: Sentinel-5P - TROPOMI


import xarray as xr
import os

# ------------------------------------------------------------
# 1. DEFINIR LA RUTA DEL ARCHIVO
# ------------------------------------------------------------

ruta_nc = "/content/S5P_NRTI_L2__NO2____20260812T025026_20260812T025526_45750_03_020901_20260812T042943.nc"

# Comprobamos que el archivo existe antes de abrirlo
if not os.path.exists(ruta_nc):
    raise FileNotFoundError(
        f"No se encuentra el archivo:\n{ruta_nc}"
    )

print("📂 Archivo encontrado correctamente.")
print(f"   {ruta_nc}")

# ------------------------------------------------------------
# 2. ABRIR EL GRUPO PRODUCT
# ------------------------------------------------------------
# Los productos Sentinel-5P L2 contienen los datos científicos
# principales dentro del grupo llamado "PRODUCT".

ds = xr.open_dataset(
    ruta_nc,
    group="PRODUCT",
    engine="h5netcdf"
)

# ------------------------------------------------------------
# 3. MOSTRAR LA INFORMACIÓN DEL DATASET
# ------------------------------------------------------------

print("\n🛰️ PRODUCTO SENTINEL-5P ABIERTO CORRECTAMENTE")
print("=" * 60)

print(ds)

In [ ]:
# ============================================================
# extraer datos principales de NO₂
# Satélite: Sentinel-5P TROPOMI
# Proyecto: Análisis del Eclipse Solar de España 2026
# Developer: Lavinia Bacaru (GitHub : @codinglavinia)
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1. EXTRAER LAS VARIABLES PRINCIPALES
# ------------------------------------------------------------

# Columna troposférica de dióxido de nitrógeno (NO₂)
no2_data = ds["nitrogendioxide_tropospheric_column"]

# Coordenadas geográficas de cada píxel
latitud = ds["latitude"]
longitud = ds["longitude"]

# Indicador de calidad de cada píxel
qa_value = ds["qa_value"]

# ------------------------------------------------------------
# 2. MOSTRAR INFORMACIÓN DE LAS VARIABLES
# ------------------------------------------------------------

print("🧪 DATOS NO₂ EXTRAÍDOS")
print("=" * 60)

print(f"📊 NO₂:")
print(f"   Dimensiones: {no2_data.dims}")
print(f"   Tipo de dato: {no2_data.dtype}")

print(f"\n🌍 Latitud:")
print(f"   Dimensiones: {latitud.dims}")

print(f"\n🌍 Longitud:")
print(f"   Dimensiones: {longitud.dims}")

print(f"\n✅ Quality flag:")
print(f"   Dimensiones: {qa_value.dims}")

# ------------------------------------------------------------
# 3. ESTADÍSTICAS BÁSICAS DEL NO₂
# ------------------------------------------------------------

print("\n📈 ESTADÍSTICAS INICIALES DEL NO₂")
print("=" * 60)

print(f"   Mínimo: {float(no2_data.min(skipna=True).values):.6e}")
print(f"   Máximo: {float(no2_data.max(skipna=True).values):.6e}")
print(f"   Media:  {float(no2_data.mean(skipna=True).values):.6e}")

In [ ]:
# ============================================================
# Diagnostico de los datos NO₂
# Satélite: Sentinel-5P TROPOMI
# Proyecto: Análisis del Eclipse Solar de España 2026
# Developer: Lavinia Bacaru (GitHub: @codinglavinia)
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1. CONVERTIR LOS DATOS NO₂ A UN ARRAY DE NUMPY
# ------------------------------------------------------------

no2_array = no2_data.values

# Contamos valores válidos y valores NaN
numero_total = no2_array.size
numero_validos = np.count_nonzero(np.isfinite(no2_array))
numero_nan = np.count_nonzero(np.isnan(no2_array))

print("🧪 DIAGNÓSTICO DE NO₂")
print("=" * 60)

print(f"📊 Número total de píxeles: {numero_total}")
print(f"✅ Valores válidos:        {numero_validos}")
print(f"❌ Valores NaN:             {numero_nan}")

# ------------------------------------------------------------
# 2. COMPROBAR LOS VALORES DEL QUALITY FLAG
# ------------------------------------------------------------

qa_array = qa_value.values

qa_validos = qa_array[np.isfinite(qa_array)]

print("\n✅ QUALITY FLAG")
print("=" * 60)

print(f"📊 Valores QA disponibles: {qa_validos.size}")

if qa_validos.size > 0:
    print(f"   Mínimo QA: {qa_validos.min()}")
    print(f"   Máximo QA: {qa_validos.max()}")
    print(f"   Media QA:  {qa_validos.mean()}")
else:
    print("❌ No hay valores QA válidos.")

# ------------------------------------------------------------
# 3. COMPROBAR LAS COORDENADAS
# ------------------------------------------------------------

lat_array = latitud.values
lon_array = longitud.values

lat_validas = lat_array[np.isfinite(lat_array)]
lon_validas = lon_array[np.isfinite(lon_array)]

print("\n🌍 COORDENADAS")
print("=" * 60)

if lat_validas.size > 0 and lon_validas.size > 0:

    print(f"Latitud mínima:  {lat_validas.min():.4f}")
    print(f"Latitud máxima:  {lat_validas.max():.4f}")

    print(f"Longitud mínima: {lon_validas.min():.4f}")
    print(f"Longitud máxima: {lon_validas.max():.4f}")

else:
    print("❌ No hay coordenadas válidas.")

In [ ]:
# ============================================================
# cargar el segundo dataset desde SENTINEL-5P
# ============================================================

from google.colab import files
import os

# ------------------------------------------------------------
# 1. SUBIR EL NUEVO ARCHIVO NETCDF
# ------------------------------------------------------------

print("🛰️ SELECCIONA EL SEGUNDO PRODUCTO SENTINEL-5P")
print("=" * 60)

archivos_subidos_2 = files.upload()

# ------------------------------------------------------------
# 2. IDENTIFICAR EL ARCHIVO CARGADO
# ------------------------------------------------------------

for nombre_archivo in archivos_subidos_2:

    ruta_archivo_2 = os.path.join(
        "/content",
        nombre_archivo
    )

    tamaño_mb = os.path.getsize(ruta_archivo_2) / (1024 * 1024)

    print("\n✅ PRODUCTO SENTINEL-5P #2 CARGADO")
    print("=" * 60)

    print(f"📄 Archivo: {nombre_archivo}")
    print(f"📂 Ruta: {ruta_archivo_2}")
    print(f"📦 Tamaño: {tamaño_mb:.2f} MB")
    print(f"✅ Existe: {os.path.exists(ruta_archivo_2)}")

In [ ]:
# ============================================================
# ABRIR EL SEGUNDO PRODUCTO SENTINEL-5P
# Satélite: Sentinel-5P
# Instrumento: TROPOMI
# Proyecto: Análisis del Eclipse Solar de España 2026
# Developer: Lavinia Bacaru (GitHub: @codinglavinia)
# ============================================================

import xarray as xr
import os

# ------------------------------------------------------------
# 1. DEFINIR LA RUTA DEL PRODUCTO #2
# ------------------------------------------------------------

ruta_nc_2 = "/content/S5P_NRTI_L2__NO2____20260812T025026_20260812T025526_45750_03_020901_20260812T042943.nc"

# Comprobar que el archivo existe
if not os.path.exists(ruta_nc_2):
    raise FileNotFoundError(
        f"No se encuentra el archivo:\n{ruta_nc_2}"
    )

print("📂 Archivo encontrado correctamente.")
print(f"   {ruta_nc_2}")

# ------------------------------------------------------------
# 2. ABRIR EL GRUPO PRODUCT
# ------------------------------------------------------------

ds_2 = xr.open_dataset(
    ruta_nc_2,
    group="PRODUCT",
    engine="h5netcdf"
)

# ------------------------------------------------------------
# 3. MOSTRAR LA ESTRUCTURA DEL DATASET
# ------------------------------------------------------------

print("\n🛰️ PRODUCTO SENTINEL-5P #2 ABIERTO CORRECTAMENTE")
print("=" * 60)

print(ds_2)

In [ ]:
# ============================================================
# Mapa del footprint del Satélite: Sentinel-5P TROPOMI
# Proyecto: Análisis del Eclipse Solar de España 2026
# Developer: Lavinia Bacaru (GitHub: @codinglavinia)
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# 1. EXTRAER COORDENADAS
# ------------------------------------------------------------

lat = ds_2["latitude"].values[0]
lon = ds_2["longitude"].values[0]

# ------------------------------------------------------------
# 2. CREAR LA MALLA DE OBSERVACIÓN
# ------------------------------------------------------------

plt.figure(figsize=(12, 8))

plt.scatter(
    lon,
    lat,
    s=2,
    alpha=0.5
)

# ------------------------------------------------------------
# 3. CONFIGURACIÓN DEL MAPA
# ------------------------------------------------------------

plt.xlabel("Longitud (°)")
plt.ylabel("Latitud (°)")

plt.title(
    "Footprint de observación — Sentinel-5P TROPOMI\n"
    "12 agosto 2026"
)

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FOOTPRINT SENTINEL-5P + ESPAÑA
# Satélite: Sentinel-5P TROPOMI
# Proyecto: Análisis del Eclipse Solar de España 2026
# Developer: Lavinia Bacaru (GitHub: @codinglavinia)
# ============================================================

import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

# ------------------------------------------------------------
# 1. COORDENADAS DEL PRODUCTO SENTINEL-5P
# ------------------------------------------------------------

lat = ds_2["latitude"].values[0]
lon = ds_2["longitude"].values[0]

# ------------------------------------------------------------
# 2. CREAR UN CONTORNO APROXIMADO DE ESPAÑA
# ------------------------------------------------------------
# Utilizamos un rectángulo geográfico únicamente para
# comprobar visualmente si el producto cubre España.

espana_lon = [-9.5, 3.5, 3.5, -9.5, -9.5]
espana_lat = [35.8, 35.8, 43.8, 43.8, 35.8]

# ------------------------------------------------------------
# 3. CREAR EL MAPA
# ------------------------------------------------------------

plt.figure(figsize=(13, 8))

# Footprint Sentinel-5P
plt.scatter(
    lon,
    lat,
    s=2,
    alpha=0.5,
    label="Footprint Sentinel-5P"
)

# Rectángulo aproximado de España
plt.plot(
    espana_lon,
    espana_lat,
    linewidth=2,
    label="España (AOI aproximada)"
)

# ------------------------------------------------------------
# 4. CONFIGURACIÓN
# ------------------------------------------------------------

plt.xlabel("Longitud (°)")
plt.ylabel("Latitud (°)")

plt.title(
    "Sentinel-5P TROPOMI — Footprint y España\n"
    "12 agosto 2026"
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3️⃣ Exploración de productos TIFF NO₂

Análisis de productos NO₂ en formato GeoTIFF, recortados a España, incluyendo superposición con la geometría del eclipse.

In [ ]:
# ============================================================
# cargar y analizar archivo .TIFF del Satélite SENTINEL-5P NO₂
# Proyecto: Análisis del Eclipse Solar de España 2026
# Developer: Lavinia Bacaru (GitHub: @codinglavinia)
# ============================================================

from google.colab import files
import os

# ------------------------------------------------------------
# 1. SUBIR EL ARCHIVO TIFF
# ------------------------------------------------------------

print("🛰️ Selecciona el TIFF de Sentinel-5P NO₂")
print("=" * 60)

archivos_tiff = files.upload()

# ------------------------------------------------------------
# 2. IDENTIFICAR EL ARCHIVO
# ------------------------------------------------------------

for nombre_tiff in archivos_tiff:

    ruta_tiff = os.path.join("/content", nombre_tiff)

    tamaño_mb = os.path.getsize(ruta_tiff) / (1024 * 1024)

    print("\n✅ TIFF CARGADO CORRECTAMENTE")
    print("=" * 60)

    print(f"📄 Archivo: {nombre_tiff}")
    print(f"📂 Ruta: {ruta_tiff}")
    print(f"📦 Tamaño: {tamaño_mb:.2f} MB")
    print(f"✅ Existe: {os.path.exists(ruta_tiff)}")

In [ ]:
# ============================================================
# Inspeccionar el TIFF de NO₂
# Satélite: Sentinel-5P TROPOMI
# Proyecto: Análisis del Eclipse Solar de España 2026
# Developer: Lavinia Bacaru (GitHub: @codinglavinia)
# ============================================================

import rasterio
import numpy as np

# ------------------------------------------------------------
# 1. Abrir el archivo TIFF
# ------------------------------------------------------------

with rasterio.open(ruta_tiff) as src:

    print("🛰️ Información del TIFF Sentinel-5P")
    print("=" * 60)

    print(f"📄 Nombre: {src.name}")
    print(f"📐 Ancho: {src.width} píxeles")
    print(f"📐 Alto: {src.height} píxeles")
    print(f"🧮 Bandas: {src.count}")
    print(f"🌍 CRS: {src.crs}")
    print(f"📍 Bounds: {src.bounds}")
    print(f"🚫 NoData: {src.nodata}")
    print(f"🔢 Tipo de dato: {src.dtypes[0]}")

    # --------------------------------------------------------
    # 2. Leer los valores del raster
    # --------------------------------------------------------

    datos_no2 = src.read(1)

    # Convertimos los valores NoData en NaN
    if src.nodata is not None:
        datos_no2 = np.where(
            datos_no2 == src.nodata,
            np.nan,
            datos_no2
        )

    # --------------------------------------------------------
    # 3. Calcular estadísticas básicas de NO₂
    # --------------------------------------------------------

    valores_validos = datos_no2[np.isfinite(datos_no2)]

    print("\n📊 Estadísticas de NO₂")
    print("=" * 60)

    print(f"Valores totales: {datos_no2.size}")
    print(f"Valores válidos: {valores_validos.size}")

    if valores_validos.size > 0:

        print(f"Mínimo: {np.min(valores_validos):.6e}")
        print(f"Máximo: {np.max(valores_validos):.6e}")
        print(f"Media:  {np.mean(valores_validos):.6e}")

    else:

        print("❌ No hay valores NO₂ válidos en el TIFF.")

In [ ]:
# ============================================================
# Inspeccionar las bandas del archivo .TIFF
# Satélite: Sentinel-5P TROPOMI
# Proyecto: Análisis del Eclipse Solar de España 2026
# Developer: Lavinia Bacaru (GitHub: @codinglavinia)
# ============================================================

import rasterio
import numpy as np

with rasterio.open(ruta_tiff) as src:

    print("🛰️ Información de las bandas")
    print("=" * 60)

    for numero_banda in range(1, src.count + 1):

        banda = src.read(numero_banda)

        valores_validos = banda[np.isfinite(banda)]

        print(f"\n📊 Banda {numero_banda}")
        print(f"   Tipo de dato: {src.dtypes[numero_banda - 1]}")
        print(f"   Valores totales: {banda.size}")
        print(f"   Valores válidos: {valores_validos.size}")

        if valores_validos.size > 0:
            print(f"   Mínimo: {valores_validos.min():.6e}")
            print(f"   Máximo: {valores_validos.max():.6e}")
            print(f"   Media:  {valores_validos.mean():.6e}")

In [ ]:


import warnings
import rasterio
import matplotlib.pyplot as plt
import numpy as np

# Ocultar temporalmente los avisos de GDAL/Rasterio
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Abrir el TIFF
# ------------------------------------------------------------

with rasterio.open(ruta_tiff) as src:

    no2 = src.read(1)
    bounds = src.bounds

# ------------------------------------------------------------
# 2. Preparar los datos
# ------------------------------------------------------------

no2 = np.where(
    np.isfinite(no2),
    no2,
    np.nan
)

print(" Raster leído correctamente")
print(f"Dimensiones: {no2.shape}")
print(f" Extensión: {bounds}")

# ------------------------------------------------------------
# 3. Crear el mapa
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 8))

imagen = ax.imshow(
    no2,
    extent=[
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ],
    origin="upper"
)

# Barra de color
fig.colorbar(
    imagen,
    ax=ax,
    label="Columna troposférica de NO₂"
)

ax.set_title(
    "Distribución de NO₂ — Sentinel-5P TROPOMI\n"
    "12 de agosto de 2026"
)

ax.set_xlabel("Longitud (°)")
ax.set_ylabel("Latitud (°)")

ax.grid(
    True,
    alpha=0.3
)

plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# Superponer la geometría del eclipse sobre el raster de NO₂
# Proyecto: Análisis del Eclipse Solar de España 2026
# ============================================================

import rasterio
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# 1. Leer el raster de NO₂
# ------------------------------------------------------------

with rasterio.open(ruta_tiff) as src:

    no2 = src.read(1)
    bounds = src.bounds

# Convertir valores no válidos en NaN
no2 = np.where(
    np.isfinite(no2),
    no2,
    np.nan
)

# ------------------------------------------------------------
# 2. Crear la figura
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 9))

# Raster de NO₂
imagen = ax.imshow(
    no2,
    extent=[
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ],
    origin="upper"
)

# ------------------------------------------------------------
# 3. Superponer la geometría de la eclipse
# ------------------------------------------------------------

gdf_penumbra.plot(
    ax=ax,
    facecolor="none",
    edgecolor="orange",
    linewidth=2,
    label="Límites de la penumbra"
)

gdf_umbra.plot(
    ax=ax,
    color="black",
    linewidth=2,
    label="Trayectoria de la umbra"
)

gdf_maximo.plot(
    ax=ax,
    color="red",
    markersize=70,
    marker="*",
    label="Greatest Eclipse Point"
)

# ------------------------------------------------------------
# 4. Barra de color
# ------------------------------------------------------------

fig.colorbar(
    imagen,
    ax=ax,
    label="Columna troposférica de NO₂"
)

# ------------------------------------------------------------
# 5. Configuración del mapa
# ------------------------------------------------------------

ax.set_title(
    "NO₂ y trayectoria del eclipse solar\n"
    "Sentinel-5P TROPOMI — España, 12 de agosto de 2026"
)

ax.set_xlabel("Longitud (°)")
ax.set_ylabel("Latitud (°)")

ax.grid(
    True,
    alpha=0.3
)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import xarray as xr

# Ruta del producto Sentinel-5P #1
ruta_nc = "/content/S5P_NRTI_L2__NO2____20260812T025026_20260812T025526_45750_03_020901_20260812T042943.nc"

# Abrimos el producto
ds = xr.open_dataset(ruta_nc)

print("Dataset cargado correctamente")
print()
print("Variables del dataset:")
print(list(ds.data_vars))

print()
print("Dimensiones:")
print(ds.dims)

print()
print("Coordenadas:")
print(list(ds.coords))

In [ ]:
import h5py

# Inspeccionamos la estructura interna del producto Sentinel-5P
with h5py.File(ruta_nc, "r") as archivo:
    print("Estructura interna del producto:")
    archivo.visititems(
        lambda nombre, objeto: print(nombre)
        if isinstance(objeto, h5py.Group)
        else None
    )

In [ ]:
import xarray as xr

# abrimos el grupo principal del producto Sentinel-5P
ds = xr.open_dataset(
    ruta_nc,
    group="PRODUCT",
    engine="h5netcdf"
)

print("producto cargado correctamente")
print("\nvariables del dataset:")
print(list(ds.data_vars))

print("\ndimensiones:")
print(ds.dims)

print("\ncoordenadas:")
print(list(ds.coords))

In [ ]:
import numpy as np

print("diagnóstico de los datos de no₂")

# extraemos las variables principales
no2 = ds["nitrogendioxide_tropospheric_column"]
qa = ds["qa_value"]

print("\nno₂:")
print(no2)

print("\nqa_value:")
print(qa)

# convertimos los datos a arrays de numpy
no2_values = no2.values
qa_values = qa.values

print("\nestadísticas de no₂:")
print("mínimo:", np.nanmin(no2_values))
print("máximo:", np.nanmax(no2_values))
print("media:", np.nanmean(no2_values))
print("valores válidos:", np.sum(np.isfinite(no2_values)))
print("valores nan:", np.sum(np.isnan(no2_values)))

print("\nestadísticas de qa_value:")
print("mínimo:", np.nanmin(qa_values))
print("máximo:", np.nanmax(qa_values))
print("media:", np.nanmean(qa_values))
print("valores válidos:", np.sum(np.isfinite(qa_values)))
print("valores nan:", np.sum(np.isnan(qa_values)))

In [ ]:
# analizamos los píxeles que contienen valores válidos de No₂

mascara_validos = np.isfinite(no2_values)

no2_validos = no2_values[mascara_validos]
qa_de_validos = qa_values[mascara_validos]

print("píxeles con no₂ válido:", len(no2_validos))

print("\nqa_value de los píxeles con no₂ válido:")
print("mínimo:", np.min(qa_de_validos))
print("máximo:", np.max(qa_de_validos))
print("media:", np.mean(qa_de_validos))
print("mediana:", np.median(qa_de_validos))

print("\nvalores de no₂ válidos:")
print("mínimo:", np.min(no2_validos))
print("máximo:", np.max(no2_validos))
print("media:", np.mean(no2_validos))
print("mediana:", np.median(no2_validos))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# seleccionar las variables de no₂ y calidad
no2_data = ds["nitrogendioxide_tropospheric_column"].squeeze()
qa_data = ds["qa_value"].squeeze()

# identificar los píxeles con valores no₂ válidos
mask_valid = np.isfinite(no2_data.values)

# extraer no₂ y qa solamente para los píxeles válidos
no2_valid = no2_data.values[mask_valid]
qa_valid = qa_data.values[mask_valid]

print("diagnóstico de los píxeles no₂ válidos")
print("--------------------------------------")
print(f"píxeles no₂ válidos: {len(no2_valid)}")
print(f"píxeles no₂ nan: {np.isnan(no2_data.values).sum()}")

print("\nqa de los píxeles no₂ válidos")
print("--------------------------------")
print(f"qa mínimo: {np.nanmin(qa_valid):.6f}")
print(f"qa máximo: {np.nanmax(qa_valid):.6f}")
print(f"qa medio:  {np.nanmean(qa_valid):.6f}")

# comprobar cuántos píxeles tienen qa superior a diferentes umbrales
for umbral in [0.0, 0.01, 0.05, 0.1, 0.5]:
    cantidad = np.sum(qa_valid >= umbral)
    print(f"qa >= {umbral}: {cantidad} píxeles")

# obtener las coordenadas geográficas de los píxeles válidos
lat_data = ds["latitude"].squeeze().values
lon_data = ds["longitude"].squeeze().values

lat_valid = lat_data[mask_valid]
lon_valid = lon_data[mask_valid]

print("\nextensión geográfica de los píxeles válidos")
print("--------------------------------------------")
print(f"latitud mínima:  {np.nanmin(lat_valid):.4f}")
print(f"latitud máxima:  {np.nanmax(lat_valid):.4f}")
print(f"longitud mínima: {np.nanmin(lon_valid):.4f}")
print(f"longitud máxima: {np.nanmax(lon_valid):.4f}")

### diagnóstico de los datos de No₂

En este paso se identifican los píxeles que contienen valores válidos de NO₂ y se analiza la calidad de estos datos mediante `qa_value`.

El producto contiene 135.000 píxeles en total. En el diagnóstico se identifican **6.096 píxeles con valores válidos de NO₂** y 128.904 valores `NaN`.

También se comprueba la distribución de `qa_value` asociada a los píxeles válidos y la extensión geográfica de las observaciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# diagnóstico de los datos de no₂
#
# en este paso se identifican los píxeles que contienen valores válidos
# de no₂ y se analiza la calidad de estos datos mediante qa_value.
#
# el producto contiene 135.000 píxeles en total.
#
# también se comprueba la extensión geográfica de las observaciones
# para determinar la cobertura espacial del producto sentinel-5p.
#
# los resultados se utilizarán para evaluar si el producto es adecuado
# para el análisis espacial del eclipse solar del 12 de agosto de 2026.


# seleccionar las variables de no₂ y calidad
no2_data = ds["nitrogendioxide_tropospheric_column"].squeeze()
qa_data = ds["qa_value"].squeeze()

# identificar los píxeles con valores no₂ válidos
mask_valid = np.isfinite(no2_data.values)

# extraer no₂ y qa solamente para los píxeles válidos
no2_valid = no2_data.values[mask_valid]
qa_valid = qa_data.values[mask_valid]

print("diagnóstico de los píxeles no₂ válidos")
print("--------------------------------------")
print(f"píxeles no₂ válidos: {len(no2_valid)}")
print(f"píxeles no₂ nan: {np.isnan(no2_data.values).sum()}")

print("\nqa de los píxeles no₂ válidos")
print("--------------------------------")
print(f"qa mínimo: {np.nanmin(qa_valid):.6f}")
print(f"qa máximo: {np.nanmax(qa_valid):.6f}")
print(f"qa medio:  {np.nanmean(qa_valid):.6f}")

# comprobar cuántos píxeles tienen qa superior a diferentes umbrales
for umbral in [0.0, 0.01, 0.05, 0.1, 0.5]:
    cantidad = np.sum(qa_valid >= umbral)
    print(f"qa >= {umbral}: {cantidad} píxeles")

# obtener las coordenadas geográficas de los píxeles válidos
lat_data = ds["latitude"].squeeze().values
lon_data = ds["longitude"].squeeze().values

lat_valid = lat_data[mask_valid]
lon_valid = lon_data[mask_valid]

print("\nextensión geográfica de los píxeles válidos")
print("--------------------------------------------")
print(f"latitud mínima:  {np.nanmin(lat_valid):.4f}")
print(f"latitud máxima:  {np.nanmax(lat_valid):.4f}")
print(f"longitud mínima: {np.nanmin(lon_valid):.4f}")
print(f"longitud máxima: {np.nanmax(lon_valid):.4f}")

In [ ]:
import os
import glob

# inventario de los productos sentinel-5p disponibles en colab
#
# se buscan todos los archivos netcdf (.nc) relacionados con sentinel-5p
# para identificar los productos disponibles para los días 11, 12 y 13
# de agosto de 2026.

archivos_nc = sorted(
    glob.glob("/content/*.nc")
)

print("productos sentinel-5p encontrados")
print("----------------------------------")

if len(archivos_nc) == 0:
    print("no se encontraron archivos .nc en /content/")
else:
    for i, archivo in enumerate(archivos_nc, start=1):
        nombre = os.path.basename(archivo)
        tamaño_mb = os.path.getsize(archivo) / (1024 * 1024)

        print(f"{i}. {nombre}")
        print(f"   tamaño: {tamaño_mb:.2f} mb")
        print()

In [ ]:
# estado actual de los datos del satelite Sentinel-5P
#
# en el entorno actual de google colab se ha encontrado un único producto
# sentinel-5p para el análisis.
#
# producto/data:
# S5P_NRTI_L2__NO2____20260812T025026_20260812T025526_45750_03_020901_20260812T042943.nc
#
# periodo de adquisición: 12 de agosto de 2026
# 02:50:26 - 02:55:26 utc
#
# este producto ya ha sido diagnosticado.
# contiene 6.096 píxeles válidos de no₂, pero su cobertura geográfica
# se encuentra aproximadamente entre 59.65° y 68.80° de latitud.

# para realizar la comparación 11-12-13 de agosto será necesario
# disponer de productos sentinel-5p cuya huella espacial incluya
# españa en las fechas correspondientes.

In [ ]:
import os

# buscar todos los productos sentinel-5p disponibles en colab
#
# esta búsqueda recorre /content y sus subcarpetas.
# no se descarga ni se modifica ningún archivo.
#
# objetivo:
# identificar todos los productos Sentinel-5P disponibles
# antes de continuar con el análisis del 11, 12 y 13 de agosto.

productos_s5p = []

for raiz, carpetas, archivos in os.walk("/content"):
    for archivo in archivos:
        if archivo.startswith("S5P_") and archivo.endswith(".nc"):
            ruta = os.path.join(raiz, archivo)
            productos_s5p.append(ruta)

productos_s5p = sorted(productos_s5p)

print("productos sentinel-5p encontrados")
print("----------------------------------")

if not productos_s5p:
    print("no se encontraron productos sentinel-5p.")
else:
    for i, ruta in enumerate(productos_s5p, start=1):
        tamaño_mb = os.path.getsize(ruta) / (1024 * 1024)

        print(f"{i}. {os.path.basename(ruta)}")
        print(f"   ruta: {ruta}")
        print(f"   tamaño: {tamaño_mb:.2f} mb")
        print()

print(f"total de productos encontrados: {len(productos_s5p)}")

In [ ]:
from google.colab import files

# cargar los productos sentinel-5p disponibles en el ordenador
#
# se utilizarán únicamente archivos que ya hemos descargado.
# no se realiza ninguna descarga nueva.

archivos_subidos = files.upload()

print("\narchivos cargados:")
print("------------------")

for nombre in archivos_subidos:
    print(nombre)

In [ ]:
import rasterio
import os

# analizar la estructura del producto TIFF del 13 de agosto
ruta_tiff = "/content/2026-08-13-00_00_2026-08-13-23_59_Sentinel-5P_NO2_NO2_(Raw).tiff"

print("Diagnóstico del producto Sentinel-5P NO₂ - 13 agosto 2026")
print("-------------------------------------------------------")

print(f"Archivo: {os.path.basename(ruta_tiff)}")
print(f"Existe: {os.path.exists(ruta_tiff)}")

with rasterio.open(ruta_tiff) as src:

    print("\nInformación raster:")
    print(f"CRS: {src.crs}")
    print(f"Ancho: {src.width} píxeles")
    print(f"Alto: {src.height} píxeles")
    print(f"Número de bandas: {src.count}")
    print(f"Tipo de datos: {src.dtypes}")
    print(f"Resolución: {src.res}")

    print("\nExtensión geográfica:")
    print(f"left:   {src.bounds.left}")
    print(f"right:  {src.bounds.right}")
    print(f"bottom: {src.bounds.bottom}")
    print(f"top:    {src.bounds.top}")

    print("\nMetadatos:")
    print(src.meta)

In [ ]:
import rasterio
import numpy as np

ruta_tiff = "/content/2026-08-13-00_00_2026-08-13-23_59_Sentinel-5P_NO2_NO2_(Raw).tiff"

with rasterio.open(ruta_tiff) as src:
    no2_tiff = src.read(1)

# diagnóstico de los valores del raster
valores_validos = no2_tiff[np.isfinite(no2_tiff)]

print("Diagnóstico de los valores NO₂ del TIFF")
print("----------------------------------------")

print(f"Total de píxeles: {no2_tiff.size}")
print(f"Píxeles válidos: {len(valores_validos)}")
print(f"Píxeles NaN: {np.isnan(no2_tiff).sum()}")

print("\nEstadísticas NO₂:")
print("-----------------")
print(f"Mínimo:  {np.min(valores_validos):.10f}")
print(f"Máximo:  {np.max(valores_validos):.10f}")
print(f"Media:   {np.mean(valores_validos):.10f}")
print(f"Mediana: {np.median(valores_validos):.10f}")

print("\nValores negativos:")
print("------------------")
print(f"Cantidad: {np.sum(valores_validos < 0)}")

print("\nValores cero:")
print("--------------")
print(f"Cantidad: {np.sum(valores_validos == 0)}")

In [ ]:
import rasterio
import matplotlib.pyplot as plt

ruta_tiff = "/content/2026-08-13-00_00_2026-08-13-23_59_Sentinel-5P_NO2_NO2_(Raw).tiff"

with rasterio.open(ruta_tiff) as src:
    no2_tiff = src.read(1)
    extent = [
        src.bounds.left,
        src.bounds.right,
        src.bounds.bottom,
        src.bounds.top
    ]

plt.figure(figsize=(12, 7))

plt.imshow(
    no2_tiff,
    extent=extent,
    origin="upper"
)

plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("Sentinel-5P NO₂ — 13 agosto 2026")

plt.colorbar(label="NO₂")
plt.show()

In [ ]:
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt

# ruta del producto TIFF
ruta_tiff = "/content/2026-08-13-00_00_2026-08-13-23_59_Sentinel-5P_NO2_NO2_(Raw).tiff"

# cargar el raster
with rasterio.open(ruta_tiff) as src:
    no2_tiff = src.read(1)
    extent = [
        src.bounds.left,
        src.bounds.right,
        src.bounds.bottom,
        src.bounds.top
    ]
    crs_tiff = src.crs

print("CRS del TIFF:", crs_tiff)

# descargar/cargar las fronteras administrativas de Natural Earth
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"

world = gpd.read_file(url)

# seleccionar España
spain = world[world["ADMIN"] == "Spain"].copy()

# asegurar que España utiliza el mismo CRS que el TIFF
spain = spain.to_crs(crs_tiff)

# crear la visualización
fig, ax = plt.subplots(figsize=(12, 8))

# raster NO₂
im = ax.imshow(
    no2_tiff,
    extent=extent,
    origin="upper"
)

# frontera de España
spain.boundary.plot(
    ax=ax,
    linewidth=1.5
)

ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_title("Sentinel-5P NO₂ — 13 agosto 2026\nCobertura espacial sobre España")

plt.colorbar(im, ax=ax, label="NO₂")
plt.show()

In [ ]:
import rasterio
import geopandas as gpd
import rasterio.mask
import numpy as np

# ruta del producto TIFF
ruta_tiff = "/content/2026-08-13-00_00_2026-08-13-23_59_Sentinel-5P_NO2_NO2_(Raw).tiff"

# cargar las fronteras administrativas
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

# seleccionar España
spain = world[world["ADMIN"] == "Spain"].copy()

with rasterio.open(ruta_tiff) as src:

    # asegurar que España está en el mismo CRS que el raster
    spain = spain.to_crs(src.crs)

    # recortar el raster utilizando la geometría de España
    no2_spain, transform_spain = rasterio.mask.mask(
        src,
        spain.geometry,
        crop=True,
        filled=False
    )

# extraer la banda
no2_spain = no2_spain[0]

# obtener únicamente los valores válidos
no2_spain_valid = no2_spain.compressed()

print("Diagnóstico NO₂ — España")
print("------------------------")

print(f"Píxeles NO₂ dentro de España: {len(no2_spain_valid)}")

print("\nEstadísticas:")
print("------------------------")
print(f"Mínimo:              {np.min(no2_spain_valid):.10f}")
print(f"Máximo:              {np.max(no2_spain_valid):.10f}")
print(f"Media:               {np.mean(no2_spain_valid):.10f}")
print(f"Mediana:             {np.median(no2_spain_valid):.10f}")
print(f"Desviación estándar: {np.std(no2_spain_valid):.10f}")

print("\nPercentiles:")
print("------------------------")
print(f"P05: {np.percentile(no2_spain_valid, 5):.10f}")
print(f"P25: {np.percentile(no2_spain_valid, 25):.10f}")
print(f"P75: {np.percentile(no2_spain_valid, 75):.10f}")
print(f"P95: {np.percentile(no2_spain_valid, 95):.10f}")

print("\nValores negativos:")
print("------------------------")
print(f"Cantidad: {np.sum(no2_spain_valid < 0)}")

In [ ]:
from google.colab import files
import os
import numpy as np
import rasterio

# ============================================================
# CARGAR EL TIFF RAW Sentinel-5P NO₂ — 11 Agosto 2026
# ============================================================

print("Selecciona el archivo TIFF RAW de NO₂ del 11 de agosto.")

archivos_cargados = files.upload()

print("\nARCHIVO CARGADO")
print("----------------")

for nombre in archivos_cargados:
    tamaño_mb = os.path.getsize(nombre) / (1024 * 1024)

    print(f"Archivo: {nombre}")
    print(f"Tamaño: {tamaño_mb:.2f} MB")

# ============================================================
# Identificar el archivo cargado
# ============================================================

ruta_tiff_11 = list(archivos_cargados.keys())[0]

print("\nPRODUCTO SELECCIONADO")
print("--------------------")
print(ruta_tiff_11)

# ============================================================
# Diagnóstico espacial del raster
# ============================================================

with rasterio.open(ruta_tiff_11) as src:

    print("\nINFORMACIÓN RASTER")
    print("------------------")
    print(f"CRS: {src.crs}")
    print(f"Ancho: {src.width} píxeles")
    print(f"Alto: {src.height} píxeles")
    print(f"Número de bandas: {src.count}")
    print(f"Tipo de datos: {src.dtypes}")
    print(f"Resolución: {src.res}")

    print("\nEXTENSIÓN GEOGRÁFICA")
    print("--------------------")
    print(f"left:   {src.bounds.left}")
    print(f"right:  {src.bounds.right}")
    print(f"bottom: {src.bounds.bottom}")
    print(f"top:    {src.bounds.top}")

    print("\nMETADATOS")
    print("---------")
    print(src.meta)

    # Leer la primera banda
    no2_11 = src.read(1).astype("float32")

# ============================================================
# Diagnóstico de los valores NO₂
# ============================================================

mask_valid = np.isfinite(no2_11)
no2_valid = no2_11[mask_valid]

print("\nDIAGNÓSTICO DE LOS VALORES NO₂ — 11 AGOSTO")
print("-------------------------------------------")

print(f"Total de píxeles: {no2_11.size}")
print(f"Píxeles válidos: {len(no2_valid)}")
print(f"Píxeles NaN: {np.isnan(no2_11).sum()}")

if len(no2_valid) > 0:

    print("\nESTADÍSTICAS NO₂")
    print("----------------")
    print(f"Mínimo:  {np.min(no2_valid):.10f}")
    print(f"Máximo:  {np.max(no2_valid):.10f}")
    print(f"Media:   {np.mean(no2_valid):.10f}")
    print(f"Mediana: {np.median(no2_valid):.10f}")

    negativos = np.sum(no2_valid < 0)
    ceros = np.sum(no2_valid == 0)

    print("\nVALORES NEGATIVOS")
    print("-----------------")
    print(f"Cantidad: {negativos}")

    print("\nVALORES CERO")
    print("------------")
    print(f"Cantidad: {ceros}")

In [ ]:
from google.colab import files
import os

# Cargar varios productos Sentinel-5P de una sola vez
archivos_cargados = files.upload()

print("\nARCHIVOS CARGADOS")
print("-----------------")

for nombre, datos in archivos_cargados.items():
    tamaño_mb = len(datos) / (1024 * 1024)

    print(f"{nombre}")
    print(f"tamaño: {tamaño_mb:.2f} MB")
    print()

In [ ]:
import os
import xarray as xr

# ============================================================
# IDENTIFICACIÓN DEL PRODUCTO PRINCIPAL — 12 AGOSTO 2026
# ============================================================

ruta_nc_12 = "/content/S5P_OFFL_L2__NO2____20260812T114331_20260812T132501_45756_03_020901_20260814T041712.nc"

print("PRODUCTO SENTINEL-5P — 12 AGOSTO 2026")
print("--------------------------------------")

print(f"Archivo: {os.path.basename(ruta_nc_12)}")
print(f"Existe: {os.path.exists(ruta_nc_12)}")

if os.path.exists(ruta_nc_12):
    tamaño_mb = os.path.getsize(ruta_nc_12) / (1024 * 1024)
    print(f"Tamaño: {tamaño_mb:.2f} MB")

In [ ]:
# ============================================================
# INSPECCIÓN DEL PRODUCTO SENTINEL-5P — 12 AGOSTO 2026
# ============================================================

import xarray as xr

print("ABRIENDO PRODUCTO SENTINEL-5P")
print("-----------------------------")

ds_12 = xr.open_dataset(
    ruta_nc_12,
    group="PRODUCT"
)

print("VARIABLES DISPONIBLES")
print("---------------------")
print(list(ds_12.data_vars))

print("\nDIMENSIONES")
print("----------")
print(ds_12.sizes)

print("\nCOORDENADAS")
print("-----------")
print(list(ds_12.coords))

In [ ]:
import numpy as np

# ============================================================
# extension geográfica del sátelite  SENTINEL-5P
# ============================================================

lat = ds_12["latitude"].values
lon = ds_12["longitude"].values

print("extensión geográfica — SENTINEL-5P")
print("----------------------------------")

print(f"Latitud mínima:  {np.nanmin(lat):.4f}")
print(f"Latitud máxima:  {np.nanmax(lat):.4f}")
print(f"Longitud mínima: {np.nanmin(lon):.4f}")
print(f"Longitud máxima: {np.nanmax(lon):.4f}")

In [ ]:
import xarray as xr

# abrir producto Sentinel-5P del 12 de agosto de 2026

print("Abriendo producto Sentinel-5P")
print("-----------------------------")

# variables disponibles en el producto
print("Variables disponibles")
print("----------------------")
print(list(ds_12.data_vars))

# dimensiones del producto
print("\nDimensiones")
print("-----------")
print(ds_12.sizes)

# coordenadas disponibles
print("\nCoordenadas")
print("----------")
print(list(ds_12.coords))

In [ ]:
import numpy as np

# seleccionar NO₂ y calidad
no2 = ds_12["nitrogendioxide_tropospheric_column"].squeeze()
qa = ds_12["qa_value"].squeeze()

# identificar píxeles NO₂ válidos
mask_valid = np.isfinite(no2.values)

no2_valid = no2.values[mask_valid]
qa_valid = qa.values[mask_valid]

print("Diagnóstico NO₂ — Sentinel-5P")
print("-----------------------------")

print(f"Píxeles totales: {no2.values.size}")
print(f"Píxeles NO₂ válidos: {len(no2_valid)}")
print(f"Píxeles NO₂ NaN: {np.isnan(no2.values).sum()}")

print("\nEstadísticas NO₂")
print("----------------")
print(f"Mínimo:  {np.min(no2_valid):.10f}")
print(f"Máximo:  {np.max(no2_valid):.10f}")
print(f"Media:   {np.mean(no2_valid):.10f}")
print(f"Mediana: {np.median(no2_valid):.10f}")

print("\nQA de los píxeles NO₂ válidos")
print("------------------------------")
print(f"Mínimo:  {np.nanmin(qa_valid):.6f}")
print(f"Máximo:  {np.nanmax(qa_valid):.6f}")
print(f"Media:   {np.nanmean(qa_valid):.6f}")
print(f"Mediana: {np.nanmedian(qa_valid):.6f}")

In [ ]:
import numpy as np

# obtener coordenadas de los píxeles
lat = ds_12["latitude"].squeeze().values
lon = ds_12["longitude"].squeeze().values

# seleccionar NO₂
no2 = ds_12["nitrogendioxide_tropospheric_column"].squeeze().values

# identificar píxeles con NO₂ válido
mask_valid = np.isfinite(no2)

lat_valid = lat[mask_valid]
lon_valid = lon[mask_valid]

print("Extensión geográfica de los píxeles NO₂ válidos")
print("----------------------------------------------")

print(f"Latitud mínima:  {np.nanmin(lat_valid):.4f}")
print(f"Latitud máxima:  {np.nanmax(lat_valid):.4f}")
print(f"Longitud mínima: {np.nanmin(lon_valid):.4f}")
print(f"Longitud máxima: {np.nanmax(lon_valid):.4f}")

# comprobar cuántos píxeles válidos están aproximadamente sobre España
mask_espana = (
    (lat_valid >= 35.0) &
    (lat_valid <= 44.0) &
    (lon_valid >= -10.0) &
    (lon_valid <= 4.5)
)

no2_espana = no2[mask_valid][mask_espana]

print("\nPíxeles NO₂ sobre el área de España")
print("------------------------------------")
print(f"Píxeles: {len(no2_espana)}")

if len(no2_espana) > 0:
    print(f"Mínimo:  {np.min(no2_espana):.10f}")
    print(f"Máximo:  {np.max(no2_espana):.10f}")
    print(f"Media:   {np.mean(no2_espana):.10f}")
    print(f"Mediana: {np.median(no2_espana):.10f}")

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# cargar la geometría de España
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)

espana = world[world["NAME"] == "Spain"].to_crs("EPSG:4326")

# crear puntos a partir de los píxeles NO₂ válidos
puntos = gpd.GeoDataFrame(
    {
        "no2": no2[mask_valid],
        "latitude": lat_valid,
        "longitude": lon_valid
    },
    geometry=[
        Point(lon_, lat_)
        for lon_, lat_ in zip(lon_valid, lat_valid)
    ],
    crs="EPSG:4326"
)

# seleccionar solamente los píxeles dentro de España
puntos_espana = gpd.sjoin(
    puntos,
    espana[["geometry"]],
    predicate="within",
    how="inner"
)

no2_espana = puntos_espana["no2"].to_numpy()

print("NO₂ dentro de España")
print("--------------------")
print(f"Píxeles: {len(no2_espana)}")

if len(no2_espana) > 0:
    print(f"Mínimo:  {np.min(no2_espana):.10f}")
    print(f"Máximo:  {np.max(no2_espana):.10f}")
    print(f"Media:   {np.mean(no2_espana):.10f}")
    print(f"Mediana: {np.median(no2_espana):.10f}")

In [ ]:
# seleccionar QA de los píxeles dentro de España
qa = ds_12["qa_value"].squeeze().values

qa_valid = qa[mask_valid]
qa_espana = qa_valid[puntos_espana.index.to_numpy()]

print("QA de los píxeles NO₂ dentro de España")
print("---------------------------------------")

print(f"Píxeles: {len(qa_espana)}")
print(f"Mínimo:  {np.nanmin(qa_espana):.6f}")
print(f"Máximo:  {np.nanmax(qa_espana):.6f}")
print(f"Media:   {np.nanmean(qa_espana):.6f}")
print(f"Mediana: {np.nanmedian(qa_espana):.6f}")

# comprobar algunos niveles de calidad
print("\nDistribución por niveles de QA")
print("------------------------------")

for umbral in [0.3, 0.5, 0.75, 0.8, 0.9]:
    cantidad = np.sum(qa_espana >= umbral)
    porcentaje = cantidad / len(qa_espana) * 100

    print(
        f"QA >= {umbral:.2f}: "
        f"{cantidad} píxeles ({porcentaje:.2f}%)"
    )

In [ ]:
# aplicar el filtro de calidad QA >= 0.75
umbral_qa = 0.75

mask_qa = qa_espana >= umbral_qa

no2_espana_qa = no2_espana[mask_qa]
qa_espana_filtrado = qa_espana[mask_qa]

print("NO₂ en España con QA >= 0.75")
print("-----------------------------")

print(f"Píxeles: {len(no2_espana_qa)}")
print(f"Porcentaje: {len(no2_espana_qa) / len(no2_espana) * 100:.2f}%")

print("\nEstadísticas NO₂")
print("----------------")
print(f"Mínimo:  {np.min(no2_espana_qa):.10f}")
print(f"Máximo:  {np.max(no2_espana_qa):.10f}")
print(f"Media:   {np.mean(no2_espana_qa):.10f}")
print(f"Mediana: {np.median(no2_espana_qa):.10f}")
print(f"Desv. estándar: {np.std(no2_espana_qa):.10f}")

print("\nPercentiles")
print("-----------")
print(f"P05: {np.percentile(no2_espana_qa, 5):.10f}")
print(f"P25: {np.percentile(no2_espana_qa, 25):.10f}")
print(f"P75: {np.percentile(no2_espana_qa, 75):.10f}")
print(f"P95: {np.percentile(no2_espana_qa, 95):.10f}")

print("\nValores negativos")
print("-----------------")
print(f"Cantidad: {np.sum(no2_espana_qa < 0)}")

In [ ]:
# conservar las coordenadas de los píxeles filtrados en los puntos de España
lat_espana = puntos_espana["latitude"].to_numpy()
lon_espana = puntos_espana["longitude"].to_numpy()

lat_espana_qa = lat_espana[mask_qa]
lon_espana_qa = lon_espana[mask_qa]

print("Coordenadas de los píxeles NO₂ filtrados")
print("-----------------------------------------")

print(f"Píxeles: {len(no2_espana_qa)}")
print(f"Latitud mínima:  {np.min(lat_espana_qa):.4f}")
print(f"Latitud máxima:  {np.max(lat_espana_qa):.4f}")
print(f"Longitud mínima: {np.min(lon_espana_qa):.4f}")
print(f"Longitud máxima: {np.max(lon_espana_qa):.4f}")

In [ ]:
import matplotlib.pyplot as plt

# visualizar la distribución espacial del NO₂
plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    lon_espana_qa,
    lat_espana_qa,
    c=no2_espana_qa,
    s=3
)

plt.colorbar(scatter, label="NO₂ (mol/m²)")

plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("NO₂ sobre España — Sentinel-5P — 12 agosto 2026")

plt.grid(True, alpha=0.3)
plt.show()

## 4️⃣ Análisis final reproducible (Google Drive) — 11, 12 y 13 de agosto

Esta es la sección **canónica** del análisis: carga de los tres productos desde Google Drive, filtrado por calidad (`QA ≥ 0.75`), recorte a la geometría real de España (no un bounding box aproximado), y cálculo de estadísticas por fecha.

In [ ]:
from google.colab import drive

# conectar Google Drive
drive.mount("/content/drive")

In [ ]:
import os

# buscar productos Sentinel-5P en Google Drive

productos_s5p = []

for raiz, carpetas, archivos in os.walk("/content/drive/MyDrive"):
    for archivo in archivos:
        if archivo.startswith("S5P_") and archivo.endswith(".nc"):
            productos_s5p.append(os.path.join(raiz, archivo))

productos_s5p = sorted(productos_s5p)

print("PRODUCTOS SENTINEL-5P EN GOOGLE DRIVE")
print("--------------------------------------")

if not productos_s5p:
    print("No se encontraron productos Sentinel-5P.")
else:
    for i, ruta in enumerate(productos_s5p, start=1):
        tamaño_mb = os.path.getsize(ruta) / (1024 * 1024)

        print(f"{i}. {os.path.basename(ruta)}")
        print(f"   tamaño: {tamaño_mb:.2f} MB")
        print(f"   ruta: {ruta}")
        print()

    print(f"Total de productos encontrados: {len(productos_s5p)}")

In [ ]:
import os
import xarray as xr

# producto Sentinel-5P — 12 agosto 2026

ruta_nc_12 = "/content/drive/MyDrive/Eclipse-2026/Eclipse -12 Agosto 2026_Copernicus Data Space-Sátelite Sentinel 5P/S5P_OFFL_L2__NO2____20260812T114331_20260812T132501_45756_03_020901_20260814T041712.nc"

print("ABRIENDO PRODUCTO SENTINEL-5P")
print("-----------------------------")

print(f"Archivo: {os.path.basename(ruta_nc_12)}")
print(f"Existe: {os.path.exists(ruta_nc_12)}")

if os.path.exists(ruta_nc_12):
    tamaño_mb = os.path.getsize(ruta_nc_12) / (1024 * 1024)
    print(f"Tamaño: {tamaño_mb:.2f} MB")

    ds = xr.open_dataset(ruta_nc_12, group="PRODUCT")

    print("\nPRODUCTO CARGADO")
    print("----------------")
    print("Variables disponibles:")
    print(list(ds.data_vars))

In [ ]:
import numpy as np

# seleccionar variables Sentinel-5P

no2_data = ds["nitrogendioxide_tropospheric_column"].squeeze().values
qa_data = ds["qa_value"].squeeze().values

lat_data = ds["latitude"].squeeze().values
lon_data = ds["longitude"].squeeze().values

# seleccionar píxeles válidos

mask_valid = (
    np.isfinite(no2_data) &
    np.isfinite(qa_data) &
    np.isfinite(lat_data) &
    np.isfinite(lon_data)
)

no2_valid = no2_data[mask_valid]
qa_valid = qa_data[mask_valid]
lat_valid = lat_data[mask_valid]
lon_valid = lon_data[mask_valid]

print("DIAGNÓSTICO DE LOS PÍXELES NO₂")
print("------------------------------")

print(f"Píxeles totales: {no2_data.size}")
print(f"Píxeles NO₂ válidos: {len(no2_valid)}")
print(f"Píxeles NO₂ NaN: {np.isnan(no2_data).sum()}")

print("\nQA de los píxeles NO₂ válidos")
print("------------------------------")

print(f"Mínimo: {np.min(qa_valid):.6f}")
print(f"Máximo: {np.max(qa_valid):.6f}")
print(f"Media:   {np.mean(qa_valid):.6f}")
print(f"Mediana: {np.median(qa_valid):.6f}")

In [ ]:
# delimitar España y aplicar el filtro QA

mask_espana = (
    (lat_valid >= 35.0) &
    (lat_valid <= 44.0) &
    (lon_valid >= -10.0) &
    (lon_valid <= 4.0)
)

no2_espana = no2_valid[mask_espana]
qa_espana = qa_valid[mask_espana]

lat_espana = lat_valid[mask_espana]
lon_espana = lon_valid[mask_espana]

# aplicar QA >= 0.75

mask_qa = qa_espana >= 0.75

no2_espana_qa = no2_espana[mask_qa]
qa_espana_qa = qa_espana[mask_qa]

lat_espana_qa = lat_espana[mask_qa]
lon_espana_qa = lon_espana[mask_qa]

print("NO₂ dentro de España")
print("--------------------")

print(f"Píxeles: {len(no2_espana)}")

print("\nNO₂ en España con QA >= 0.75")
print("-----------------------------")

print(f"Píxeles: {len(no2_espana_qa)}")
print(f"Porcentaje: {len(no2_espana_qa) / len(no2_espana) * 100:.2f}%")

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# cargar frontera real de España

url_spain = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"

mundo = gpd.read_file(url_spain)

espana = mundo[mundo["ADMIN"] == "Spain"].to_crs("EPSG:4326")

print("GEOMETRÍA DE ESPAÑA")
print("-------------------")
print(f"País encontrado: {len(espana)}")
print(f"CRS: {espana.crs}")

# crear puntos de los píxeles válidos
puntos = gpd.GeoSeries(
    [Point(lon, lat) for lon, lat in zip(lon_valid, lat_valid)],
    crs="EPSG:4326"
)

# identificar los píxeles que están realmente dentro de España
mask_espana_real = puntos.within(
    espana.geometry.union_all()
).values

no2_espana = no2_valid[mask_espana_real]
qa_espana = qa_valid[mask_espana_real]

lat_espana = lat_valid[mask_espana_real]
lon_espana = lon_valid[mask_espana_real]

# aplicar QA >= 0.75
mask_qa = qa_espana >= 0.75

no2_espana_qa = no2_espana[mask_qa]
qa_espana_qa = qa_espana[mask_qa]

lat_espana_qa = lat_espana[mask_qa]
lon_espana_qa = lon_espana[mask_qa]

print("\nNO₂ dentro de España")
print("--------------------")
print(f"Píxeles: {len(no2_espana)}")

print("\nNO₂ en España con QA >= 0.75")
print("-----------------------------")
print(f"Píxeles: {len(no2_espana_qa)}")
print(f"Porcentaje: {len(no2_espana_qa) / len(no2_espana) * 100:.2f}%")

In [ ]:
# identificar los valores altos de NO₂

p95 = np.percentile(no2_espana_qa, 95)

mask_altos = no2_espana_qa >= p95

print("Valores altos de NO₂ — España")
print("-----------------------------")

print(f"Umbral P95: {p95:.10f}")
print(f"Píxeles >= P95: {np.sum(mask_altos)}")

print("\nExtensión de las zonas con valores altos")
print("-----------------------------------------")

print(f"Latitud mínima:  {np.min(lat_espana_qa[mask_altos]):.4f}")
print(f"Latitud máxima:  {np.max(lat_espana_qa[mask_altos]):.4f}")
print(f"Longitud mínima: {np.min(lon_espana_qa[mask_altos]):.4f}")
print(f"Longitud máxima: {np.max(lon_espana_qa[mask_altos]):.4f}")

In [ ]:
import os

# buscar TIFF de NO₂ en Google Drive

tiff_no2 = []

for raiz, carpetas, archivos in os.walk("/content/drive/MyDrive"):
    for archivo in archivos:
        if archivo.lower().endswith((".tif", ".tiff")) and "no2" in archivo.lower():
            tiff_no2.append(os.path.join(raiz, archivo))

print("TIFF NO₂ EN GOOGLE DRIVE")
print("------------------------")

for i, ruta in enumerate(sorted(tiff_no2), start=1):
    tamaño_mb = os.path.getsize(ruta) / (1024 * 1024)

    print(f"{i}. {os.path.basename(ruta)}")
    print(f"   tamaño: {tamaño_mb:.2f} MB")
    print(f"   ruta: {ruta}")
    print()

print(f"Total: {len(tiff_no2)}")

In [ ]:
import os
import numpy as np
import rasterio

# producto Sentinel-5P — 11 agosto 2026

ruta_tiff_11 = "/content/drive/MyDrive/Eclipse-2026/Antes del Eclipse-  11 Agosto 2026_Copernicus Data Space-Sátelite Sentinel 5P/2026-08-11-00_00_2026-08-11-23_59_Sentinel-5P_NO2_NO2_(Raw).tiff"

print("ABRIENDO PRODUCTO SENTINEL-5P")
print("-----------------------------")

print(f"Archivo: {os.path.basename(ruta_tiff_11)}")
print(f"Existe: {os.path.exists(ruta_tiff_11)}")

if os.path.exists(ruta_tiff_11):

    tamaño_mb = os.path.getsize(ruta_tiff_11) / (1024 * 1024)
    print(f"Tamaño: {tamaño_mb:.2f} MB")

    with rasterio.open(ruta_tiff_11) as src:

        print("\nINFORMACIÓN RASTER")
        print("------------------")
        print(f"CRS: {src.crs}")
        print(f"Ancho: {src.width} píxeles")
        print(f"Alto: {src.height} píxeles")
        print(f"Bandas: {src.count}")
        print(f"Tipo de datos: {src.dtypes}")
        print(f"Resolución: {src.res}")

        print("\nEXTENSIÓN GEOGRÁFICA")
        print("--------------------")
        print(f"left:   {src.bounds.left}")
        print(f"right:  {src.bounds.right}")
        print(f"bottom: {src.bounds.bottom}")
        print(f"top:    {src.bounds.top}")

        no2_11 = src.read(1).astype("float32")

print("\nPRODUCTO 11 AGOSTO CARGADO")
print("---------------------------")
print(f"Píxeles totales: {no2_11.size}")

In [ ]:
# identificar las bandas del producto-data satelital Sentinel 5P desde 11 Agosto 2026

with rasterio.open(ruta_tiff_11) as src:

    print("BANDAS DEL TIFF — 11 AGOSTO")
    print("---------------------------")

    for i in range(1, src.count + 1):
        print(f"Banda {i}")
        print(f"  descripción: {src.descriptions[i - 1]}")
        print(f"  dtype: {src.dtypes[i - 1]}")
        print(f"  estadísticas: {src.statistics(i)}")
        print()

In [ ]:
# seleccionar NO₂ y QA del TIFF

with rasterio.open(ruta_tiff_11) as src:
    no2_11 = src.read(1).astype("float32")
    qa_11 = src.read(2).astype("float32")

    transform_11 = src.transform
    bounds_11 = src.bounds

# crear coordenadas de los píxeles

filas, columnas = np.indices(no2_11.shape)

longitudes = (
    transform_11.c
    + (columnas + 0.5) * transform_11.a
)

latitudes = (
    transform_11.f
    + (filas + 0.5) * transform_11.e
)

# píxeles válidos

mask_valid_11 = (
    np.isfinite(no2_11) &
    np.isfinite(qa_11)
)

print("NO₂ Y QA — 11 AGOSTO")
print("--------------------")

print(f"Píxeles totales: {no2_11.size}")
print(f"Píxeles válidos: {np.sum(mask_valid_11)}")
print(f"Píxeles NaN: {np.sum(~mask_valid_11)}")

In [ ]:
# delimitar España con la geometría real

from shapely.geometry import Point

puntos_11 = gpd.GeoSeries(
    [
        Point(lon, lat)
        for lon, lat in zip(
            longitudes[mask_valid_11],
            latitudes[mask_valid_11]
        )
    ],
    crs="EPSG:4326"
)

mask_espana_11 = puntos_11.within(
    espana.geometry.union_all()
).values

# seleccionar NO₂ y QA dentro de España

no2_espana_11 = no2_11[mask_valid_11][mask_espana_11]
qa_espana_11 = qa_11[mask_valid_11][mask_espana_11]

lat_espana_11 = latitudes[mask_valid_11][mask_espana_11]
lon_espana_11 = longitudes[mask_valid_11][mask_espana_11]

print("NO₂ dentro de España — 11 agosto")
print("---------------------------------")

print(f"Píxeles: {len(no2_espana_11)}")

In [ ]:
# aplicar filtro QA >= 0.75

mask_qa_11 = qa_espana_11 >= 0.75

no2_espana_11_qa = no2_espana_11[mask_qa_11]
qa_espana_11_qa = qa_espana_11[mask_qa_11]

lat_espana_11_qa = lat_espana_11[mask_qa_11]
lon_espana_11_qa = lon_espana_11[mask_qa_11]

print("NO₂ en España con QA >= 0.75 — 11 agosto")
print("------------------------------------------")

print(f"Píxeles: {len(no2_espana_11_qa)}")
print(f"Porcentaje: {len(no2_espana_11_qa) / len(no2_espana_11) * 100:.2f}%")

print("\nQA de los píxeles seleccionados")
print("--------------------------------")

print(f"Mínimo:  {np.min(qa_espana_11_qa):.6f}")
print(f"Máximo:  {np.max(qa_espana_11_qa):.6f}")
print(f"Media:   {np.mean(qa_espana_11_qa):.6f}")
print(f"Mediana: {np.median(qa_espana_11_qa):.6f}")

In [ ]:
# estadísticas NO₂ — 11 agosto

print("NO₂ en España con QA >= 0.75 — 11 agosto")
print("------------------------------------------")

print(f"Píxeles: {len(no2_espana_11_qa)}")

print("\nEstadísticas NO₂")
print("----------------")

print(f"Mínimo:              {np.min(no2_espana_11_qa):.10f}")
print(f"Máximo:              {np.max(no2_espana_11_qa):.10f}")
print(f"Media:               {np.mean(no2_espana_11_qa):.10f}")
print(f"Mediana:             {np.median(no2_espana_11_qa):.10f}")
print(f"Desviación estándar: {np.std(no2_espana_11_qa):.10f}")

print("\nPercentiles")
print("-----------")

print(f"P05: {np.percentile(no2_espana_11_qa, 5):.10f}")
print(f"P25: {np.percentile(no2_espana_11_qa, 25):.10f}")
print(f"P75: {np.percentile(no2_espana_11_qa, 75):.10f}")
print(f"P95: {np.percentile(no2_espana_11_qa, 95):.10f}")

print("\nValores negativos")
print("-----------------")

print(f"Cantidad: {np.sum(no2_espana_11_qa < 0)}")

In [ ]:
# valores altos de NO₂ — 11 agosto

p95_11 = np.percentile(no2_espana_11_qa, 95)

mask_altos_11 = no2_espana_11_qa >= p95_11

print("Valores altos de NO₂ — España — 11 agosto")
print("-------------------------------------------")

print(f"Umbral P95: {p95_11:.10f}")
print(f"Píxeles >= P95: {np.sum(mask_altos_11)}")

print("\nExtensión de las zonas con valores altos")
print("-----------------------------------------")

print(f"Latitud mínima:  {np.min(lat_espana_11_qa[mask_altos_11]):.4f}")
print(f"Latitud máxima:  {np.max(lat_espana_11_qa[mask_altos_11]):.4f}")
print(f"Longitud mínima: {np.min(lon_espana_11_qa[mask_altos_11]):.4f}")
print(f"Longitud máxima: {np.max(lon_espana_11_qa[mask_altos_11]):.4f}")

In [ ]:
import os
import numpy as np
import rasterio

# producto Sentinel-5P — 13 agosto 2026

ruta_tiff_13 = "/content/drive/MyDrive/Eclipse-2026/Post Eclipse- 13 Agosto 2026__Copernicus Data Space-Sátelite Sentinel 5P/2026-08-13-00_00_2026-08-13-23_59_Sentinel-5P_NO2_NO2_(Raw).tiff"

print("ABRIENDO PRODUCTO SENTINEL-5P")
print("-----------------------------")

print(f"Archivo: {os.path.basename(ruta_tiff_13)}")
print(f"Existe: {os.path.exists(ruta_tiff_13)}")

if os.path.exists(ruta_tiff_13):

    tamaño_mb = os.path.getsize(ruta_tiff_13) / (1024 * 1024)
    print(f"Tamaño: {tamaño_mb:.2f} MB")

    with rasterio.open(ruta_tiff_13) as src:

        print("\nINFORMACIÓN RASTER")
        print("------------------")
        print(f"CRS: {src.crs}")
        print(f"Ancho: {src.width} píxeles")
        print(f"Alto: {src.height} píxeles")
        print(f"Bandas: {src.count}")
        print(f"Tipo de datos: {src.dtypes}")
        print(f"Resolución: {src.res}")

        print("\nEXTENSIÓN GEOGRÁFICA")
        print("--------------------")
        print(f"left:   {src.bounds.left}")
        print(f"right:  {src.bounds.right}")
        print(f"bottom: {src.bounds.bottom}")
        print(f"top:    {src.bounds.top}")

        print("\nBANDAS")
        print("------")

        for i in range(1, src.count + 1):
            print(f"Banda {i}: {src.dtypes[i - 1]}")

In [ ]:
# diagnóstico NO₂ — 13 agosto

with rasterio.open(ruta_tiff_13) as src:
    no2_13 = src.read(1).astype("float32")
    transform_13 = src.transform

# coordenadas de los píxeles

filas_13, columnas_13 = np.indices(no2_13.shape)

longitudes_13 = (
    transform_13.c
    + (columnas_13 + 0.5) * transform_13.a
)

latitudes_13 = (
    transform_13.f
    + (filas_13 + 0.5) * transform_13.e
)

# píxeles válidos

mask_valid_13 = np.isfinite(no2_13)

no2_valid_13 = no2_13[mask_valid_13]

print("DIAGNÓSTICO NO₂ — 13 AGOSTO")
print("---------------------------")

print(f"Píxeles totales: {no2_13.size}")
print(f"Píxeles NO₂ válidos: {len(no2_valid_13)}")
print(f"Píxeles NO₂ NaN: {np.sum(~mask_valid_13)}")

print("\nEstadísticas NO₂")
print("----------------")

print(f"Mínimo:  {np.min(no2_valid_13):.10f}")
print(f"Máximo:  {np.max(no2_valid_13):.10f}")
print(f"Media:   {np.mean(no2_valid_13):.10f}")
print(f"Mediana: {np.median(no2_valid_13):.10f}")

print("\nValores negativos")
print("-----------------")
print(f"Cantidad: {np.sum(no2_valid_13 < 0)}")

In [ ]:
# seleccionar los píxeles NO₂ válidos que se encuentran dentro de la geometría real de España

puntos_13 = gpd.GeoSeries(
    [
        Point(lon, lat)
        for lon, lat in zip(
            longitudes_13[mask_valid_13],
            latitudes_13[mask_valid_13]
        )
    ],
    crs="EPSG:4326"
)

# identificar los píxeles cuya posición geográfica está dentro del territorio español

mask_espana_13 = puntos_13.within(
    espana.geometry.union_all()
).values

# extraer los valores NO₂ correspondientes al territorio de España

no2_espana_13 = no2_valid_13[mask_espana_13]

# conservar las coordenadas de los píxeles seleccionados para el análisis espacial posterior

lat_espana_13 = latitudes_13[mask_valid_13][mask_espana_13]
lon_espana_13 = longitudes_13[mask_valid_13][mask_espana_13]

print("NO₂ dentro de España — 13 agosto")
print("---------------------------------")

print(f"Píxeles: {len(no2_espana_13)}")

print("\nEstadísticas NO₂")
print("----------------")

print(f"Mínimo:  {np.min(no2_espana_13):.10f}")
print(f"Máximo:  {np.max(no2_espana_13):.10f}")
print(f"Media:   {np.mean(no2_espana_13):.10f}")
print(f"Mediana: {np.median(no2_espana_13):.10f}")

In [ ]:
# calcular el percentil 95 para identificar los valores altos de NO₂ en España

p95_13 = np.percentile(no2_espana_13, 95)

# seleccionar los píxeles con valores de NO₂ iguales o superiores al P95

mask_altos_13 = no2_espana_13 >= p95_13

print("Valores altos de NO₂ — España — 13 agosto")
print("-------------------------------------------")

print(f"Umbral P95: {p95_13:.10f}")
print(f"Píxeles >= P95: {np.sum(mask_altos_13)}")

print("\nExtensión de las zonas con valores altos")
print("-----------------------------------------")

print(f"Latitud mínima:  {np.min(lat_espana_13[mask_altos_13]):.4f}")
print(f"Latitud máxima:  {np.max(lat_espana_13[mask_altos_13]):.4f}")
print(f"Longitud mínima: {np.min(lon_espana_13[mask_altos_13]):.4f}")
print(f"Longitud máxima: {np.max(lon_espana_13[mask_altos_13]):.4f}")

## 5️⃣ Comparación final y resultados

Tabla y gráficos comparativos de la evolución del NO₂ entre las tres fechas analizadas.

In [ ]:
import pandas as pd
import numpy as np

# comparar los principales indicadores de dióxido de nitrógeno (NO₂)
# antes, durante y después del eclipse

comparacion_no2 = {
    "Fecha": [
        "11 agosto",
        "12 agosto",
        "13 agosto"
    ],
    "Píxeles": [
        len(no2_espana_11_qa),
        len(no2_espana_qa),
        len(no2_espana_13)
    ],
    "Media de NO₂": [
        np.mean(no2_espana_11_qa),
        np.mean(no2_espana_qa),
        np.mean(no2_espana_13)
    ],
    "Mediana de NO₂": [
        np.median(no2_espana_11_qa),
        np.median(no2_espana_qa),
        np.median(no2_espana_13)
    ],
    "P95 de NO₂": [
        p95_11,
        p95,
        p95_13
    ],
    "Píxeles con valores altos": [
        np.sum(mask_altos_11),
        np.sum(mask_altos),
        np.sum(mask_altos_13)
    ]
}

df_comparacion = pd.DataFrame(comparacion_no2)

print("COMPARACIÓN DEL DIÓXIDO DE NITRÓGENO (NO₂)")
print("-------------------------------------------")

display(df_comparacion)

In [ ]:
import matplotlib.pyplot as plt

# valores de la media de dióxido de nitrógeno (NO₂) obtenidos en el análisis

fechas = ["11 agosto", "12 agosto", "13 agosto"]

media_no2 = [
    0.0000272378,
    0.0000269283,
    0.0000244301
]

plt.figure(figsize=(9, 5))

plt.plot(
    fechas,
    media_no2,
    marker="o",
    linewidth=2
)

# añadir los valores exactos sobre cada punto

for fecha, valor in zip(fechas, media_no2):
    plt.annotate(
        f"{valor:.8f}",
        (fecha, valor),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center"
    )

plt.title("Evolución de la media de dióxido de nitrógeno (NO₂)")
plt.xlabel("Fecha")
plt.ylabel("Media de NO₂ (mol/m²)")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# valores del percentil 95 de dióxido de nitrógeno (NO₂)

fechas = ["11 agosto", "12 agosto", "13 agosto"]

p95_no2 = [
    0.0000428410,
    0.0000411468,
    0.0000388586
]

plt.figure(figsize=(9, 5))

plt.plot(
    fechas,
    p95_no2,
    marker="o",
    linewidth=2
)

# mostrar los valores exactos de P95 sobre cada punto

for fecha, valor in zip(fechas, p95_no2):
    plt.annotate(
        f"{valor:.8f}",
        (fecha, valor),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center"
    )

plt.title("Evolución del P95 de dióxido de nitrógeno (NO₂)")
plt.xlabel("Fecha")
plt.ylabel("P95 de NO₂ (mol/m²)")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# calcular la variación porcentual de la media y del P95 (percentil 95)
# de dióxido de nitrógeno (NO₂)

media_11 = 0.0000272378
media_12 = 0.0000269283
media_13 = 0.0000244301

p95_11 = 0.0000428410
p95_12 = 0.0000411468
p95_13 = 0.0000388586

variacion_media_11_12 = ((media_12 - media_11) / media_11) * 100
variacion_media_12_13 = ((media_13 - media_12) / media_12) * 100
variacion_media_11_13 = ((media_13 - media_11) / media_11) * 100

variacion_p95_11_12 = ((p95_12 - p95_11) / p95_11) * 100
variacion_p95_12_13 = ((p95_13 - p95_12) / p95_12) * 100
variacion_p95_11_13 = ((p95_13 - p95_11) / p95_11) * 100

print("VARIACIÓN DEL DIÓXIDO DE NITRÓGENO (NO₂)")
print("----------------------------------------")

print("\nMedia de NO₂")
print("------------")
print(f"11 → 12 agosto: {variacion_media_11_12:.2f}%")
print(f"12 → 13 agosto: {variacion_media_12_13:.2f}%")
print(f"11 → 13 agosto: {variacion_media_11_13:.2f}%")

print("\nP95 (percentil 95) de NO₂")
print("-------------------------")
print(f"11 → 12 agosto: {variacion_p95_11_12:.2f}%")
print(f"12 → 13 agosto: {variacion_p95_12_13:.2f}%")
print(f"11 → 13 agosto: {variacion_p95_11_13:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# variación porcentual de la media y del P95 (percentil 95) de NO₂

periodos = ["11 → 12 agosto", "12 → 13 agosto"]

variacion_media = [
    -1.14,
    -9.28
]

variacion_p95 = [
    -3.95,
    -5.56
]

plt.figure(figsize=(9, 5))

plt.plot(
    periodos,
    variacion_media,
    marker="o",
    linewidth=2,
    label="Media de NO₂"
)

plt.plot(
    periodos,
    variacion_p95,
    marker="o",
    linewidth=2,
    label="P95 (percentil 95)"
)

# mostrar los valores porcentuales sobre cada punto

for periodo, valor in zip(periodos, variacion_media):
    plt.annotate(
        f"{valor:.2f}%",
        (periodo, valor),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center"
    )

for periodo, valor in zip(periodos, variacion_p95):
    plt.annotate(
        f"{valor:.2f}%",
        (periodo, valor),
        textcoords="offset points",
        xytext=(0, -15),
        ha="center"
    )

plt.axhline(0, linewidth=1)

plt.title("Variación porcentual del dióxido de nitrógeno (NO₂)")
plt.xlabel("Periodo")
plt.ylabel("Variación (%)")

plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

# 🌑🛰️ Conclusiones finales y limitaciones del análisis

##  Objetivo del análisis

El objetivo principal fue estudiar la evolución del **dióxido de nitrógeno (NO₂)** en España antes, durante y después del **eclipse solar del 12 de agosto de 2026**.

Para ello se analizaron tres fechas:

- 🌤️ **11 de agosto:** periodo previo al eclipse.
- 🌑 **12 de agosto:** día del eclipse solar.
- 🌤️ **13 de agosto:** periodo posterior al eclipse.

---

## 📊 Resultados principales

Los resultados muestran una **disminución temporal de los niveles de NO₂** durante el periodo analizado.

### 📉 Media de NO₂

- **11 → 12 agosto:** −1,14 %
- **12 → 13 agosto:** −9,28 %
- **11 → 13 agosto:** **−10,31 %**

### 📉 P95 (percentil 95) de NO₂

- **11 → 12 agosto:** −3,95 %
- **12 → 13 agosto:** −5,56 %
- **11 → 13 agosto:** **−9,30 %**

La mayor variación se produjo entre el **12 y el 13 de agosto**, es decir, después del día del eclipse.

---

## 🌑 Relación con el eclipse solar :

Los resultados muestran una **variación temporal del NO₂ coincidente con el periodo del eclipse solar del 12 de agosto de 2026**.

Sin embargo, los datos analizados por sí solos **no permiten demostrar que el eclipse haya sido la causa directa de la disminución observada**.

Por tanto, el resultado debe interpretarse como una **observación de la evolución espacial y temporal del NO₂ alrededor del eclipse**, y no como una relación causal demostrada.

---

## ⚠️ Limitaciones del análisis:

- 🛰️ Los productos utilizados no son idénticos en formato y disponibilidad de variables para todos los días.
- ✅ Para los productos en los que estaba disponible la información de calidad, se utilizó **QA ≥ 0,75**.
- 📄 El TIFF utilizado para el **13 de agosto** contiene únicamente la banda de NO₂ y no dispone de una banda QA equivalente.
- 🔢 El número de píxeles analizados no es idéntico entre todos los productos, especialmente el **12 de agosto**, debido al filtrado de calidad.
- 🌦️ Las concentraciones de NO₂ pueden estar influenciadas por las **condiciones meteorológicas**, el transporte atmosférico, las emisiones locales y la circulación de masas de aire.
- ⏱️ El periodo analizado comprende únicamente **tres días**, por lo que no permite establecer una tendencia atmosférica de largo plazo.
- 🔬 Para determinar un posible efecto específico del eclipse sería necesario incorporar **datos meteorológicos, transporte atmosférico y una serie temporal más amplia**.

---

## 🎯 Conclusión general:

En conjunto, el análisis mediante **Sentinel-5P** permite identificar una **disminución de los niveles de dióxido de nitrógeno (NO₂) sobre España entre el 11 y el 13 de agosto de 2026**.

La disminución observada coincide temporalmente con el **eclipse solar del 12 de agosto**, pero la mayor variación se produce después del eclipse.

Por ello, los resultados constituyen una **evidencia observacional de una variación temporal del NO₂ alrededor del eclipse**, pero no permiten atribuir dicha variación exclusivamente al fenómeno astronómico.

> 🌑🛰️ **El análisis demuestra cómo los datos de observación de la Tierra pueden utilizarse para estudiar posibles cambios atmosféricos asociados a acontecimientos astronómicos, combinando teledetección, análisis geoespacial y procesamiento de datos científicos.**

---

### 🚀 Próximos pasos:

- 🗺️ Visualización cartográfica de los resultados en **QGIS**.
- 🌍 Representación **3D** de la distribución espacial del NO₂.
- 🌑 Integración de la trayectoria del eclipse sobre la cartografía.
- 📊 Comparación visual de las condiciones **antes, durante y después del eclipse**.